# Final basin set: glaciers inside CAMELS-CL gauged basins

Builds, checks and saves the basin set used for all glacier-streamflow analysis (strategy C).

**How the basins are chosen**
1. Start from the CAMELS-CL basins that contain at least one RGI glacier.
2. Keep the largest basins, meaning those not contained in any other basin.
3. If the gauge of a largest basin fails the streamflow checks, use the basins nested directly inside it instead, and go one level further in where those also fail.
4. A glacier outside all usable basins is left out. Every glacier is in one basin at most.

**Expected numbers (check them when you run it):** 266 glacierized basins with 6,883 glaciers, 41 largest basins, 143 gauges passing the completeness criteria, 40 final basins with 6,577 glaciers.

**Saves to** `Output/final_basin_set/`: `final_basin_table.csv`, `glacier_assignment.csv`, `final_basins.gpkg`. Later notebooks should read these files.

Not included on purpose: the comparison of basin groupings, the nesting and glacier-fraction cutoff tests, and the Caro catchment comparison. Those stay in the archived notebooks.

## 1. Setup

In [ ]:
import os, calendar, pickle, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

BASE_PATH   = '/Users/milliespencer/Desktop/CR2_OGGM_Paper'
CAMELS_FULL = f'{BASE_PATH}/Full_CAMELS_all_basins/CAMELS_CL_v202201'
BOUNDS_PATH = f'{CAMELS_FULL}/camels_cl_boundaries/camels_cl_boundaries.shp'
Q_DAY_PATH  = f'{CAMELS_FULL}/q_m3s_day.csv'
RGI_PATH    = f'{BASE_PATH}/files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv'
RESULTS_PKL = f'{BASE_PATH}/Output/all_chile_results_v2/RESULTS.pkl'   # only used by the runoff > Q check
OUT_DIR     = f'{BASE_PATH}/Output/final_basin_set'
os.makedirs(OUT_DIR, exist_ok=True)

AREA_CRS    = 'ESRI:102033'    # South America Albers equal area, for areas
CONTAIN_MIN = 0.9              # a basin is "inside" another if >= this fraction of it is (the result does not depend on this)

# streamflow completeness criteria (same as the Methods text)
SUMMER_THRESHOLD, WINTER_THRESHOLD, LAT_BOUNDARY = 0.70, 0.50, -38.0

ZONES = ['OT3', 'DA1', 'DA2', 'DA3', 'WA1', 'WA2', 'WA3', 'WA4', 'WA5', 'WA6']
ZC = {'OT3': '#8c564b', 'DA1': '#e377c2', 'DA2': '#7f7f7f', 'DA3': '#bcbd22', 'WA1': '#17becf',
      'WA2': '#2ca02c', 'WA3': '#d62728', 'WA4': '#1f77b4', 'WA5': '#ff7f0e', 'WA6': '#9467bd'}

for p in [BOUNDS_PATH, Q_DAY_PATH, RGI_PATH, RESULTS_PKL]:
    print(f'{os.path.exists(p)}  {p}')

## 2. Load basins and glaciers

Each glacier is placed in a basin by its RGI centroid. A glacier sits in every basin that contains it (CAMELS-CL basins are cumulative, so smaller basins sit inside larger ones).

In [ ]:
# CAMELS-CL basins
bounds_all = gpd.read_file(BOUNDS_PATH)
bounds_all['gauge_id'] = bounds_all['gauge_id'].astype(str)
bounds_all['lat'] = bounds_all.to_crs(epsg=4326).geometry.centroid.y.values    # used for the 38S rule
bounds_all = bounds_all.to_crs(AREA_CRS)
bounds_all['geometry'] = bounds_all.buffer(0)
bounds_all['area_km2_poly'] = bounds_all.geometry.area / 1e6
if 'gauge_name' not in bounds_all.columns:
    bounds_all['gauge_name'] = ''
names = bounds_all.set_index('gauge_id')['gauge_name'].to_dict()

# RGI glaciers (centroids) with their zone
rgi = pd.read_csv(RGI_PATH)
gl = gpd.GeoDataFrame(rgi[['RGIId', 'Area', 'Cluster', 'TermType']].copy(),
                      geometry=gpd.points_from_xy(rgi.CenLon, rgi.CenLat),
                      crs='EPSG:4326').to_crs(AREA_CRS)
gl['Cluster'] = gl['Cluster'].astype(str)

# glacier -> every basin that contains it
pj = gpd.sjoin(gl[['Area', 'Cluster', 'geometry']],
               bounds_all[['gauge_id', 'area_km2_poly', 'geometry']],
               how='inner', predicate='within')
pj['gid'] = pj.index
pj = pj.reset_index(drop=True)
members = pj.groupby('gauge_id')['gid'].apply(list).to_dict()     # basin -> glaciers inside it

# basins that contain at least one glacier
cm = bounds_all[bounds_all['gauge_id'].isin(set(pj['gauge_id']))].copy().reset_index(drop=True)
area = cm.set_index('gauge_id')['area_km2_poly']

print(f'CAMELS-CL basins: {len(bounds_all)} | glaciers (RGI): {len(gl):,} | glacier area: {gl["Area"].sum():,.0f} km2')
print(f'Basins containing glaciers: {len(cm)} (expected 266)')
print(f'Glaciers inside at least one basin: {pj["gid"].nunique():,} (expected 6,883)')

## 3. Table 1: glaciers by zone, and how many are inside a gauged basin

"Gauged" means inside a CAMELS-CL basin, before the streamflow completeness criteria. Calving glaciers are lake-terminating or marine-terminating (RGI TermType 2 or 1).

In [ ]:
g = gl.copy()
g['gauged'] = g.index.isin(pj['gid'])
g['area_gauged'] = g['Area'].where(g['gauged'], 0)
g['calving'] = g['TermType'].isin([1, 2])

table1 = g.groupby('Cluster').agg(glaciers=('Area', 'size'),
                                  area_km2=('Area', 'sum'),
                                  gauged_glaciers=('gauged', 'sum'),
                                  gauged_area_km2=('area_gauged', 'sum'),
                                  calving_glaciers=('calving', 'sum')).reindex(ZONES)
table1.loc['All'] = table1.sum()
table1['pct_glaciers'] = 100 * table1['gauged_glaciers'] / table1['glaciers']
table1['pct_area'] = 100 * table1['gauged_area_km2'] / table1['area_km2']
print(table1.round(0).astype({'glaciers': int, 'gauged_glaciers': int, 'calving_glaciers': int}).to_string())
table1.to_csv(f'{OUT_DIR}/table1_glaciers_by_zone.csv')

## 4. Nesting between basins

An arrow a -> b means basin a is inside basin b. Largest basins are the ones with no arrow leaving them.

In [ ]:
cmv = cm[['gauge_id', 'geometry']]
pairs = gpd.overlay(cmv.rename(columns={'gauge_id': 'a'}), cmv.rename(columns={'gauge_id': 'b'}),
                    how='intersection')
pairs = pairs[pairs['a'] != pairs['b']].copy()
pairs['frac_a_in_b'] = (pairs.geometry.area / 1e6) / pairs['a'].map(area)
pairs = pairs[(pairs['frac_a_in_b'] >= CONTAIN_MIN) & (pairs['a'].map(area) < pairs['b'].map(area))]

G = nx.DiGraph()
G.add_nodes_from(cm['gauge_id'])
G.add_edges_from(zip(pairs['a'], pairs['b']))       # a -> b: a is inside b
H = nx.transitive_reduction(G)                      # direct containment only

terminal = {n for n in G.nodes if G.out_degree(n) == 0}     # largest basins
finest   = {n for n in G.nodes if G.in_degree(n) == 0}      # basins with nothing inside
print(f'Largest basins (not inside any other): {len(terminal)} (expected 41)')
print(f'Finest basins (nothing inside them):   {len(finest)}')

## 5. Streamflow completeness criteria

A year is complete if December (previous year) to April each have at least 70% valid days and at least 4 of the 7 months May to November have at least 50%. North of 38°S a basin needs 5 complete years in each of 2000-2009 and 2010-2019. South of 38°S it needs 10 complete years in 2000-2019.

In [ ]:
q_day = pd.read_csv(Q_DAY_PATH, parse_dates=['date'])
q_day = q_day[(q_day['year'] >= 1999) & (q_day['year'] <= 2019)]

gids = [g for g in cm['gauge_id'] if g in q_day.columns]
ok = q_day[gids].notna() & (q_day[gids] >= 0)
cnt = ok.groupby([q_day['year'], q_day['month']]).sum()
dim = pd.Series([calendar.monthrange(y, m)[1] for y, m in cnt.index], index=cnt.index)
frac = cnt.div(dim, axis=0)          # share of days with valid data, by (year, month) and gauge

def complete_years(g):
    f = frac[g]
    yrs = []
    for y in range(2000, 2020):
        summer = [f.get((y - 1, 12), 0)] + [f.get((y, m), 0) for m in (1, 2, 3, 4)]
        winter = [f.get((y, m), 0) for m in range(5, 12)]
        if all(s >= SUMMER_THRESHOLD for s in summer) and sum(w >= WINTER_THRESHOLD for w in winter) >= 4:
            yrs.append(y)
    return yrs

lat = cm.set_index('gauge_id')['lat']
rows = []
for g in cm['gauge_id']:
    yrs = complete_years(g) if g in gids else []
    n_pre, n_post = sum(y <= 2009 for y in yrs), sum(y >= 2010 for y in yrs)
    passes = (n_pre >= 5 and n_post >= 5) if lat[g] > LAT_BOUNDARY else len(yrs) >= 10
    rows.append(dict(gauge_id=g, n_complete_years=len(yrs), n_2000_2009=n_pre, n_2010_2019=n_post, passes=passes))
completeness = pd.DataFrame(rows).set_index('gauge_id')
completeness.to_csv(f'{OUT_DIR}/completeness_by_gauge.csv')

VALID_IDS = set(completeness.index[completeness['passes']])
print(f'Pass completeness: {len(VALID_IDS)} of {len(cm)} glacierized basins (expected 143)')
print(f'  largest basins that pass: {len(VALID_IDS & terminal)} of {len(terminal)} (expected 27 of 41)')
print('Largest basins that fail:', sorted(terminal - VALID_IDS))

## 6. Check: months where modeled glacier runoff is greater than measured streamflow

Glacier runoff should never be greater than the flow at a gauge downstream of the glaciers. This cell counts, for each gauge, the share of months where it is. Nothing is excluded unless you set `RO_THRESHOLD`.

Open decisions before setting it:
- The threshold. The old Methods said 10% of months, the hierarchy notebook used 20%. The table below shows how many basins each one would remove.
- A basin removed here is treated like one that fails the completeness criteria, so its sub-basins are used instead.
- Two gauges (5702001 and 5706001, Maipo system) were excluded by hand pending Alexis's review of their raw streamflow records. Put them in `MANUAL_EXCLUDE` to apply that.

In [ ]:
RO_THRESHOLD   = None       # None = do not apply; or 10 or 20 (% of months)
RO_DATASET     = 'CR2MET'   # dataset used to decide which basins are removed
MANUAL_EXCLUDE = set()      # e.g. {'5702001', '5706001'}

def pct_months_over(res):
    m = res['monthly']
    m = m[m['q_m3_mon'] > 0].dropna(subset=['glacier_runoff_m3', 'q_m3_mon'])
    if len(m) == 0:
        return np.nan
    return ((m['glacier_runoff_m3'] / m['q_m3_mon'] * 100) > 100).mean() * 100

ro = pd.DataFrame(index=sorted(VALID_IDS))
if os.path.exists(RESULTS_PKL):
    with open(RESULTS_PKL, 'rb') as f:
        RESULTS = pickle.load(f)
    for ds in RESULTS:
        ro[ds] = [pct_months_over(RESULTS[ds][g]) if RESULTS[ds].get(g) is not None else np.nan
                  for g in ro.index]
    print(f'Gauges with results: {int(ro.notna().any(axis=1).sum())} of {len(ro)} passing gauges')
    print('\nBasins with more than X% of months where glacier runoff > streamflow:')
    print(pd.DataFrame({f'>{t}%': (ro > t).sum() for t in (5, 10, 20, 30)}).to_string())
    print(f'\nWorst basins ({RO_DATASET}):')
    print(ro[RO_DATASET].sort_values(ascending=False).head(12).round(1).to_string())
    ro.to_csv(f'{OUT_DIR}/runoff_over_streamflow_pct_months.csv')
else:
    print('RESULTS.pkl not found: skipping the check')

screened_out = set()
if RO_THRESHOLD is not None and RO_DATASET in ro.columns:
    screened_out = set(ro.index[ro[RO_DATASET] > RO_THRESHOLD])
valid_final = (VALID_IDS - screened_out - set(MANUAL_EXCLUDE)) & set(cm['gauge_id'])
print(f'\nRemoved by the runoff > streamflow check: {len(screened_out)} | removed by hand: {len(set(MANUAL_EXCLUDE))}')
print(f'Basins usable for the analysis: {len(valid_final)}')

## 7. Final basin set (strategy C)

For each largest basin: use it if its gauge is usable, otherwise use the basins directly inside it, and so on.

In [ ]:
def pick(n):
    if n in valid_final:
        return [n]
    return [u for k in H.predecessors(n) for u in pick(k)]

units = sorted({u for t in terminal for u in pick(t)})

# glaciers in each final basin
all_ids = [g for u in units for g in members.get(u, [])]
ids = set(all_ids)
print(f'Final basins: {len(units)} (expected 40 with no runoff check)')
print(f'Glaciers covered: {len(ids):,} ({100 * len(ids) / len(gl):.1f}% of all) | '
      f'area: {gl.loc[list(ids), "Area"].sum():,.0f} km2 ({100 * gl.loc[list(ids), "Area"].sum() / gl["Area"].sum():.1f}%)')
print(f'Glaciers counted in more than one basin: {len(all_ids) - len(ids)} (must be 0)')

# one row per final basin
rows = []
for u in units:
    sub = gl.loc[members.get(u, [])]
    by_zone = sub.groupby('Cluster')['Area'].sum().sort_values(ascending=False)
    rows.append(dict(
        gauge_id=u, gauge_name=names.get(u, ''),
        largest_basin=';'.join(sorted(nx.descendants(G, u) & terminal)) or u,
        is_largest_basin=u in terminal,
        n_glaciers=len(sub), glacier_area_km2=sub['Area'].sum(), basin_area_km2=area[u],
        zone=by_zone.index[0] if len(by_zone) else '',
        zone_share_of_ice_pct=100 * by_zone.iloc[0] / by_zone.sum() if len(by_zone) else np.nan,
        zones_present=','.join(by_zone.index)))
final_table = pd.DataFrame(rows)
print(f'\nBasins used instead of a failed larger basin: {int((~final_table["is_largest_basin"]).sum())}')
print(final_table[['gauge_id', 'gauge_name', 'is_largest_basin', 'n_glaciers', 'glacier_area_km2',
                   'basin_area_km2', 'zone', 'zone_share_of_ice_pct']].round(1).to_string(index=False))

## 8. Funnel table (for the Methods)

In [ ]:
all_area_km2  = bounds_all.geometry.unary_union.area / 1e6
glac_area_all = gl['Area'].sum()

def step(label, n_basins, basin_area, gid_list):
    s = list(set(gid_list))
    return dict(step=label, basins=n_basins, basin_area_km2=round(basin_area), glaciers=len(s),
                glacier_area_km2=round(gl.loc[s, 'Area'].sum()) if s else 0)

funnel = pd.DataFrame([
    dict(step='1. RGI v6.0 glaciers and all CAMELS-CL basins', basins=len(bounds_all),
         basin_area_km2=round(all_area_km2), glaciers=len(gl), glacier_area_km2=round(glac_area_all)),
    step('2. Basins containing glaciers; glaciers inside a basin', len(cm),
         cm.geometry.unary_union.area / 1e6, pj['gid'].unique()),
    step('3. Largest basins only (before the streamflow criteria)', len(terminal),
         area[list(terminal)].sum(), [g for u in terminal for g in members.get(u, [])]),
    step('4. Final basins (after the streamflow criteria)', len(units),
         area[units].sum(), all_ids),
])
funnel['glaciers_lost_vs_previous'] = (-funnel['glaciers'].diff()).fillna(0).astype(int)
funnel['pct_of_all_glaciers'] = (100 * funnel['glaciers'] / len(gl)).round(1)
print(funnel.to_string(index=False))
funnel.to_csv(f'{OUT_DIR}/funnel_table.csv', index=False)

## 9. Do the final basins overlap?

Each row is a pair of basins that share some area. `pct_of_basin_1` and `pct_of_basin_2` are the overlap as a share of each of the two basins in the pair. The glacier check is the one that matters: no glacier should be in two basins.

In [ ]:
U = cm[cm['gauge_id'].isin(units)][['gauge_id', 'area_km2_poly', 'geometry']]
o = gpd.overlay(U.rename(columns={'gauge_id': 'basin_1', 'area_km2_poly': 'area_1'}),
                U.rename(columns={'gauge_id': 'basin_2', 'area_km2_poly': 'area_2'}),
                how='intersection')
o = o[o['basin_1'] < o['basin_2']].copy()
o['overlap_km2'] = o.geometry.area / 1e6
o = o[o['overlap_km2'] > 0]
o['pct_of_basin_1'] = 100 * o['overlap_km2'] / o['area_1']
o['pct_of_basin_2'] = 100 * o['overlap_km2'] / o['area_2']

print(f'Pairs of final basins that overlap: {len(o)}')
if len(o):
    print(f'Largest overlap: {o["overlap_km2"].max():.1f} km2 | '
          f'largest share of either basin: {max(o["pct_of_basin_1"].max(), o["pct_of_basin_2"].max()):.3f}%')
    print(o.sort_values('overlap_km2', ascending=False)
           [['basin_1', 'basin_2', 'overlap_km2', 'pct_of_basin_1', 'pct_of_basin_2']].head(10).round(4).to_string(index=False))
print(f'\nGlaciers in more than one final basin: {len(all_ids) - len(ids)} (must be 0)')

## 10. Coverage by zone

Glaciers and ice area in the final basins, by zone. Basins hold glaciers from more than one zone in some places, so the table also shows how mixed each basin is. Each glacier keeps its own zone. A basin is labeled with the zone holding most of its glacier area.

In [ ]:
cov = gl.loc[list(ids)].groupby('Cluster').agg(glaciers_covered=('Area', 'size'), area_covered_km2=('Area', 'sum'))
tot = gl.groupby('Cluster').agg(glaciers=('Area', 'size'), area_km2=('Area', 'sum'))
zone_cov = tot.join(cov).fillna(0).reindex(ZONES)
zone_cov.loc['All'] = zone_cov.sum()
zone_cov['pct_glaciers'] = 100 * zone_cov['glaciers_covered'] / zone_cov['glaciers']
zone_cov['pct_area'] = 100 * zone_cov['area_covered_km2'] / zone_cov['area_km2']
zone_cov['basins_labeled_with_zone'] = final_table['zone'].value_counts().reindex(zone_cov.index).fillna(0).astype(int)
zone_cov.loc['All', 'basins_labeled_with_zone'] = len(final_table)
print(zone_cov.round(0).astype({'glaciers': int, 'glaciers_covered': int}).to_string())

mixed = final_table[final_table['zone_share_of_ice_pct'] < 90]
print(f'\nBasins where the main zone holds less than 90% of the ice: {len(mixed)} of {len(final_table)}')
print(mixed[['gauge_id', 'gauge_name', 'zone', 'zone_share_of_ice_pct', 'zones_present', 'glacier_area_km2']]
      .round(1).to_string(index=False))
zone_cov.to_csv(f'{OUT_DIR}/coverage_by_zone.csv')

## 11. Map

Final basins colored by their main zone. Red dots are glaciers that are not in a final basin. Set `ZOOM` to a list of zones (for example `['DA2', 'WA1']`) to zoom in.

In [ ]:
ZOOM = None     # None = all of Chile

geom = cm.set_index('gauge_id').geometry
unit_gdf = gpd.GeoDataFrame(final_table[['gauge_id', 'zone']],
                            geometry=[geom[u] for u in final_table['gauge_id']], crs=AREA_CRS)
uncovered = gl.loc[gl.index.difference(list(ids))]

sel = gl[gl['Cluster'].isin(ZOOM)] if ZOOM else gl
minx, miny, maxx, maxy = sel.total_bounds
px, py = (maxx - minx) * 0.05, (maxy - miny) * 0.05

fig, ax = plt.subplots(figsize=(6, 14) if not ZOOM else (8, 10))
unit_gdf.plot(ax=ax, color=unit_gdf['zone'].map(ZC), edgecolor='k', lw=0.3, alpha=0.8)
uncovered.plot(ax=ax, color='red', markersize=0.4)
ax.set_xlim(minx - px, maxx + px)
ax.set_ylim(miny - py, maxy + py)
ax.set_axis_off()
ax.set_title(f'Final basins ({len(units)}) | glaciers covered: {len(ids):,} of {len(gl):,}')
ax.legend(handles=[Patch(facecolor=ZC[z], edgecolor='k', label=z) for z in ZONES] +
          [Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=5, label='glacier not covered')],
          title='main zone of basin', loc='lower left', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/final_basins_map.png', dpi=200, bbox_inches='tight')
plt.show()

## 12. Save

Later notebooks read these files instead of rebuilding the basin set.
- `final_basin_table.csv`: one row per final basin.
- `glacier_assignment.csv`: one row per covered glacier (RGIId, basin, zone, area, terminus type).
- `final_basins.gpkg`: basin polygons (Albers projection) with the same columns as the table.

In [ ]:
final_table.to_csv(f'{OUT_DIR}/final_basin_table.csv', index=False)

assign = []
for u in units:
    sub = gl.loc[members.get(u, []), ['RGIId', 'Cluster', 'Area', 'TermType']].copy()
    sub.insert(1, 'gauge_id', u)
    assign.append(sub)
assign = pd.concat(assign, ignore_index=True)
assign.to_csv(f'{OUT_DIR}/glacier_assignment.csv', index=False)

out_gdf = gpd.GeoDataFrame(final_table, geometry=[geom[u] for u in final_table['gauge_id']], crs=AREA_CRS)
out_gdf.to_file(f'{OUT_DIR}/final_basins.gpkg', driver='GPKG')

print(f'Saved to {OUT_DIR}:')
print(f'  final_basin_table.csv   ({len(final_table)} basins)')
print(f'  glacier_assignment.csv  ({len(assign):,} glaciers, {assign["RGIId"].nunique():,} unique)')
print(f'  final_basins.gpkg')